In [1]:
import io
import requests
import numpy as np
import pandas as pd
import yfinance as yf


In [7]:
import io
import requests
import pandas as pd

url = "https://www.iposcoop.com/ipos-recently-filed/"

# Use a browser header to bypass request blocks
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

resp = requests.get(url, headers=headers)

if resp.status_code != 200:
    print(f"Error fetching page: Status code {resp.status_code}")
else:
    tables = pd.read_html(io.StringIO(resp.text))
    
    # Automatically find the table that has our target column
    df = None
    for t in tables:
        if 'Expected To Trade' in t.columns:
            df = t
            break
            
    if df is None:
        raise ValueError("Could not find the IPO table. Check if columns changed!")

    # Filter rows for withdrawn statuses
    df_withdrawn = df[df['Expected To Trade'].str.strip() == 'Withdrawn'].copy()

    # Classify matching criteria rules sequentially
    def classify_company(name):
        if not isinstance(name, str):
            return "Other"
        if "Technologies" in name: return "Technologies"
        elif any(x in name for x in ["Acquisition Corp", "Acquisition Corporation", "Corp"]): return "Acquisition Corp"
        elif any(x in name for x in ["Inc", "Incorporated"]): return "Inc."
        elif "Group" in name: return "Group"
        elif any(x in name for x in ["Ltd", "Limited"]): return "Limited"
        elif any(x in name for x in ["Holdings", "Holding"]): return "Holdings"
        return "Other"

    # CHANGED: 'Company Name' changed to 'Company' to match the website column structure
    df_withdrawn['Company Type'] = df_withdrawn['Company'].apply(classify_company)

    # Average high and low price fields
    def parse_avg_price(row):
        try:
            low = float(str(row['Price Low']).replace('$', '').replace(',', '').strip())
            high = float(str(row['Price High']).replace('$', '').replace(',', '').strip())
            return (low + high) / 2
        except:
            return None

    df_withdrawn['Avg_price'] = df_withdrawn.apply(parse_avg_price, axis=1)
    df_withdrawn['Shares (millions)'] = pd.to_numeric(df_withdrawn['Shares (millions)'].astype(str).str.replace('$', '').str.replace(',', ''), errors='coerce')
    df_withdrawn['Est $ Vol (millions)'] = pd.to_numeric(df_withdrawn['Est $ Vol (millions)'].astype(str).str.replace('$', '').str.replace(',', ''), errors='coerce')

    # Fall back on Estimated Vol metric if shares calculation is null
    df_withdrawn['Shares_offered_value'] = (df_withdrawn['Shares (millions)'] * df_withdrawn['Avg_price']).fillna(df_withdrawn['Est $ Vol (millions)'])

    # Output the grouped aggregate values
    q1_output = df_withdrawn.groupby('Company Type')['Shares_offered_value'].sum().sort_values(ascending=False)
    print(q1_output)


Company Type
Acquisition Corp    499.9850
Inc.                351.0000
Holdings            311.6575
Other               290.4450
Limited             219.2500
Technologies        184.9000
Group                56.5750
Name: Shares_offered_value, dtype: float64


In [9]:
import io
import requests
import pandas as pd
import yfinance as yf

# 1. Update with the exact URL specified in the homework guidelines
url_2025 = "https://www.iposcoop.com/2025-pricings/"

# Provide browser headers to clear request blocking walls
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

resp_2025 = requests.get(url_2025, headers=headers)

if resp_2025.status_code != 200:
    print(f"Error fetching page: Status code {resp_2025.status_code}")
else:
    tables_2025 = pd.read_html(io.StringIO(resp_2025.text))
    
    # Automatically locate the correct data structure container 
    df_2025 = None
    for t in tables_2025:
        if 'Offer Date' in t.columns:
            df_2025 = t
            break
            
    if df_2025 is None:
        raise ValueError("Could not find the 2025 Pricing table. Verify target columns.")

    # 2. Safely parse string fields into datetimes
    df_2025['Offer Date'] = pd.to_datetime(df_2025['Offer Date'])
    
    # Clean up the return metric spaces to check active listings
    df_2025['Return'] = df_2025['Return'].astype(str).str.strip()
    filtered_ipos = df_2025[(df_2025['Offer Date'] < '2025-09-01') & (df_2025['Return'] != '0.00%') & (df_2025['Return'] != '0%')]
    
    # 3. Extract the clean tracking symbols
    tickers = filtered_ipos['Symbol'].dropna().tolist()
    print(f"Extracted {len(tickers)} symbols for download tracking.")

    # Download historical pricing panels from Yahoo Finance API
    stocks_df = yf.download(tickers, start="2025-01-01", end="2026-09-15")['Close']

    # Flatten index levels if current package models default to multi-indexing
    if isinstance(stocks_df.columns, pd.MultiIndex):
        stocks_df.columns = stocks_df.columns.get_level_values(0)
        
    print(f"Downloaded pricing metrics successfully for {len(stocks_df.columns)} active tickers.")


Extracted 146 symbols for download tracking.


[*                      3%                       ]  4 of 146 completed$MTSR: possibly delisted; no timezone found
[****                   8%                       ]  11 of 146 completedHTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: EPWK"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: AHL"}}}
[******                12%                       ]  17 of 146 completed$AHL: possibly delisted; no timezone found
[******                12%                       ]  18 of 146 completed$EPWK: possibly delisted; no timezone found
[**************        29%                       ]  42 of 146 completed$SDM: possibly delisted; no timezone found
[**************        29%                       ]  43 of 146 completed$WGRX: possibly delisted; no timezone found
[********************  41%                       ]  60 of 146 completed$AGH: possibly delisted; no timez

Downloaded pricing metrics successfully for 146 active tickers.


In [10]:
sharpe_ratios = []
for ticker in stocks_df.columns:
    col = stocks_df[ticker].dropna()
    if len(col) >= 252:
        growth_252d = col / col.shift(252)
        volatility = col.rolling(30).std() * np.sqrt(252)
        sharpe = (growth_252d - 0.05) / volatility
        if '2026-09-11' in sharpe.index:
            sharpe_ratios.append(sharpe.loc['2026-09-11'])

print("Median Sharpe Ratio:", pd.Series(sharpe_ratios).median())


Median Sharpe Ratio: 0.050142446812154115


In [11]:
holding_periods = {f'future_growth_{i}_m': i * 21 for i in range(1, 13)}
medians = {}

for name, days in holding_periods.items():
    period_returns = []
    for ticker in stocks_df.columns:
        col = stocks_df[ticker].dropna()
        if len(col) > days:
            intro_price = col.iloc[0]
            future_price = col.iloc[days]
            period_returns.append(future_price / intro_price)
    medians[name] = pd.Series(period_returns).median()

# Output table directly in Jupyter
pd.DataFrame.from_dict(medians, orient='index', columns=['Median Growth'])


,Median Growth
future_growth_1_m,0.935351
future_growth_2_m,0.892958
future_growth_3_m,0.827160
future_growth_4_m,0.730400
future_growth_5_m,0.690909
future_growth_6_m,0.726000
future_growth_7_m,0.662000
future_growth_8_m,0.603508
future_growth_9_m,0.580294
future_growth_10_m,0.533333
